<a href="https://colab.research.google.com/github/Dev-Hyper-Flix/Blockchain-powered-peer-to-peer-tutoring-marketplace./blob/main/deeplabv3%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# 1. Install Dependencies
# ==========================================
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless

import os

print("Generating project files...")

# ==========================================
# 2. Create requirements.txt
# ==========================================
with open("requirements.txt", "w") as f:
    f.write('''torch
torchvision
segmentation-models-pytorch
albumentations
opencv-python-headless
matplotlib
tqdm
numpy
''')
print("Created requirements.txt")


# ==========================================
# 3. Create utils.py
# ==========================================
with open("utils.py", "w") as f:
    f.write('''import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

def set_seed(seed=42):
    """Sets the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def plot_training_curves(train_losses, val_losses, val_ious, save_path="training_curves.png"):
    """Plots and saves the training/validation losses and validation IoU vs Epoch."""
    epochs = range(1, len(train_losses) + 1)

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Loss Plot
    ax[0].plot(epochs, train_losses, label='Train Loss', color='blue')
    ax[0].plot(epochs, val_losses, label='Validation Loss', color='red')
    ax[0].set_title('Loss vs Epochs')
    ax[0].set_xlabel('Epochs')
    ax[0].set_ylabel('Loss')
    ax[0].legend()
    ax[0].grid(True)

    # IoU Plot
    ax[1].plot(epochs, val_ious, label='Validation IoU', color='green')
    ax[1].set_title('Validation IoU vs Epochs')
    ax[1].set_xlabel('Epochs')
    ax[1].set_ylabel('IoU')
    ax[1].legend()
    ax[1].grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    print(f"Saved training curves to {save_path}")

def visualize_results(images, masks, preds, num_samples=10, save_path="test_visualizations.png"):
    """Visualizes Original Image, Ground Truth, and Predicted Mask."""
    num_samples = min(num_samples, len(images))
    if num_samples == 0:
        return

    indices = random.sample(range(len(images)), num_samples)
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

    # Handle the case where num_samples is 1
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, idx in enumerate(indices):
        # Denormalize image for visualization
        img = images[idx].transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)

        mask = masks[idx].squeeze()
        pred = preds[idx].squeeze()

        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Original Image")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].set_title("Ground Truth Mask")
        axes[i, 1].axis('off')

        axes[i, 2].imshow(pred, cmap='gray')
        axes[i, 2].set_title("Predicted Mask")
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig(save_path)
    print(f"Saved visualizations to {save_path}")
''')
print("Created utils.py")


# ==========================================
# 4. Create transforms.py
# ==========================================
with open("transforms.py", "w") as f:
    f.write('''import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transforms(image_size=512):
    """Returns transformations for the training dataset."""
    return A.Compose([
        A.Resize(height=image_size, width=image_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

def get_val_test_transforms(image_size=512):
    """Returns transformations for the validation and test datasets."""
    return A.Compose([
        A.Resize(height=image_size, width=image_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
''')
print("Created transforms.py")


# ==========================================
# 5. Create dataset.py
# ==========================================
with open("dataset.py", "w") as f:
    f.write('''import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class CrackDataset(Dataset):
    def __init__(self, root_dir, phase, transform=None):
        """
        Args:
            root_dir (str): Path to the dataset directory.
            phase (str): 'train', 'val', or 'test'.
            transform (callable, optional): Albumentations transforms.
        """
        self.images_dir = os.path.join(root_dir, phase, 'images')
        self.masks_dir = os.path.join(root_dir, phase, 'masks')
        self.transform = transform

        # Filter only image files and sort to ensure alignment by filename
        valid_extensions = ('.jpg', '.jpeg', '.png')
        if os.path.exists(self.images_dir):
            self.image_files = sorted([f for f in os.listdir(self.images_dir) if f.lower().endswith(valid_extensions)])
        else:
            self.image_files = []

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, img_name)

        # Read image in RGB
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Read mask in Grayscale
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Binary mask: White (255) = crack, Black (0) = background
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Ensure mask has shape 1 x H x W and is float32
        mask = mask.clone().detach().unsqueeze(0).to(torch.float32)

        return image, mask
''')
print("Created dataset.py")


# ==========================================
# 6. Create model.py
# ==========================================
with open("model.py", "w") as f:
    f.write('''import segmentation_models_pytorch as smp

def get_model():
    """Initializes and returns the DeepLabV3+ model with ResNet34 encoder."""
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        classes=1,
        activation=None  # We use BCEWithLogitsLoss which applies sigmoid internally
    )
    return model
''')
print("Created model.py")


# ==========================================
# 7. Create losses.py
# ==========================================
with open("losses.py", "w") as f:
    f.write('''import torch
import torch.nn as nn

class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceBCELoss, self).__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets):
        # Compute BCE Loss
        bce_loss = self.bce(inputs, targets)

        # Compute Dice Loss
        inputs_sig = torch.sigmoid(inputs)
        inputs_flat = inputs_sig.view(-1)
        targets_flat = targets.view(-1)

        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1 - ((2. * intersection + self.smooth) /
                         (inputs_flat.sum() + targets_flat.sum() + self.smooth))

        # Combined Loss
        return bce_loss + dice_loss
''')
print("Created losses.py")


# ==========================================
# 8. Create metrics.py
# ==========================================
with open("metrics.py", "w") as f:
    f.write('''import torch

class MetricTracker:
    """Manually tracks and computes classification metrics over an entire dataset."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.tp = 0.0
        self.tn = 0.0
        self.fp = 0.0
        self.fn = 0.0

    def update(self, preds, targets):
        """
        Updates the true positive, true negative, false positive, and false negative counts.
        """
        # Threshold at 0.5 after sigmoid
        preds = (torch.sigmoid(preds) > 0.5).float()
        targets = targets.float()

        self.tp += (preds * targets).sum().item()
        self.tn += ((1 - preds) * (1 - targets)).sum().item()
        self.fp += (preds * (1 - targets)).sum().item()
        self.fn += ((1 - preds) * targets).sum().item()

    def get_metrics(self):
        """Calculates IoU, Dice, Accuracy, and Precision."""
        epsilon = 1e-7

        iou = self.tp / (self.tp + self.fp + self.fn + epsilon)
        dice = (2 * self.tp) / (2 * self.tp + self.fp + self.fn + epsilon)
        accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn + epsilon)
        precision = self.tp / (self.tp + self.fp + epsilon)

        return iou, dice, accuracy, precision
''')
print("Created metrics.py")


# ==========================================
# 9. Create train.py
# ==========================================
with open("train.py", "w") as f:
    f.write('''import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import CrackDataset
from transforms import get_train_transforms, get_val_test_transforms
from model import get_model
from losses import DiceBCELoss
from metrics import MetricTracker
from utils import set_seed, plot_training_curves

def train():
    # Hyperparameters
    DATASET_DIR = "dataset"
    IMAGE_SIZE = 512
    BATCH_SIZE = 4
    EPOCHS = 50
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 2
    PIN_MEMORY = True

    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    if not os.path.exists(DATASET_DIR):
        print(f"Error: Directory '{DATASET_DIR}' not found. Please upload dataset.")
        return

    # Datasets and Dataloaders
    train_dataset = CrackDataset(DATASET_DIR, phase='train', transform=get_train_transforms(IMAGE_SIZE))
    val_dataset = CrackDataset(DATASET_DIR, phase='val', transform=get_val_test_transforms(IMAGE_SIZE))

    if len(train_dataset) == 0:
        print("No training images found! Ensure dataset structure is correct.")
        return

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # Model, Loss, Optimizer
    model = get_model().to(device)
    criterion = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    metrics_tracker = MetricTracker()

    best_iou = 0.0
    history_train_loss = []
    history_val_loss = []
    history_val_iou = []

    # Training Loop
    for epoch in range(1, EPOCHS + 1):
        # ----------------- TRAIN -----------------
        model.train()
        train_loss_epoch = 0.0

        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
        for images, masks in loop:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss_epoch += loss.item() * images.size(0)
            loop.set_postfix(loss=loss.item())

        train_loss_epoch /= len(train_loader.dataset)
        history_train_loss.append(train_loss_epoch)

        # ----------------- VALIDATION -----------------
        model.eval()
        val_loss_epoch = 0.0
        metrics_tracker.reset()

        with torch.no_grad():
            loop_val = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]")
            for images, masks in loop_val:
                images = images.to(device)
                masks = masks.to(device)

                outputs = model(images)
                loss = criterion(outputs, masks)

                val_loss_epoch += loss.item() * images.size(0)
                metrics_tracker.update(outputs, masks)

        val_loss_epoch /= len(val_loader.dataset)
        iou, dice, accuracy, precision = metrics_tracker.get_metrics()

        history_val_loss.append(val_loss_epoch)
        history_val_iou.append(iou)

        # Print metrics
        print(f"\\nEpoch {epoch}/{EPOCHS}")
        print(f"Train Loss: {train_loss_epoch:.4f} | Validation Loss: {val_loss_epoch:.4f}")
        print(f"IoU: {iou:.4f} | Dice: {dice:.4f} | Accuracy: {accuracy:.4f} | Precision: {precision:.4f}\\n")

        # Save Models
        torch.save(model.state_dict(), "last_model.pth")
        if iou > best_iou:
            best_iou = iou
            torch.save(model.state_dict(), "best_model.pth")
            print(">>> Saved best_model.pth\\n")

    # Plotting
    plot_training_curves(history_train_loss, history_val_loss, history_val_iou)
    print("Training complete.")

if __name__ == "__main__":
    train()
''')
print("Created train.py")


# ==========================================
# 10. Create test.py
# ==========================================
with open("test.py", "w") as f:
    f.write('''import os
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import CrackDataset
from transforms import get_val_test_transforms
from model import get_model
from losses import DiceBCELoss
from metrics import MetricTracker
from utils import set_seed, visualize_results

def test():
    # Hyperparameters
    DATASET_DIR = "dataset"
    IMAGE_SIZE = 512
    BATCH_SIZE = 4
    NUM_WORKERS = 2
    PIN_MEMORY = True

    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    if not os.path.exists("best_model.pth"):
        print("Error: best_model.pth not found. Please run train.py first.")
        return

    # Dataset and Dataloader
    test_dataset = CrackDataset(DATASET_DIR, phase='test', transform=get_val_test_transforms(IMAGE_SIZE))
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    if len(test_dataset) == 0:
        print("No test images found! Ensure dataset structure is correct.")
        return

    # Load Model
    model = get_model().to(device)
    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    model.eval()

    criterion = DiceBCELoss()
    metrics_tracker = MetricTracker()

    test_loss = 0.0
    all_images = []
    all_masks = []
    all_preds = []

    with torch.no_grad():
        loop = tqdm(test_loader, desc="Testing")
        for images, masks in loop:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            test_loss += loss.item() * images.size(0)
            metrics_tracker.update(outputs, masks)

            # Store data for visualization (moving to CPU)
            preds_binary = (torch.sigmoid(outputs) > 0.5).float()

            all_images.extend(images.cpu().numpy())
            all_masks.extend(masks.cpu().numpy())
            all_preds.extend(preds_binary.cpu().numpy())

    test_loss /= len(test_loader.dataset)
    iou, dice, accuracy, precision = metrics_tracker.get_metrics()

    print("\\n--- Test Results ---")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"IoU:       {iou:.4f}")
    print(f"Dice:      {dice:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print("--------------------\\n")

    # Visualizations
    print("Generating visualizations for 10 random samples...")
    visualize_results(all_images, all_masks, all_preds, num_samples=10)

if __name__ == "__main__":
    test()
''')
print("Created test.py")
print("=====================================================")
print("All files have been generated successfully!")
print("Make sure your dataset folder is uploaded to Colab.")
print("Then, create a new cell and run: !python train.py")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.8 MB/s eta 0:00:00
Generating project files...
Created requirements.txt
Created utils.py
Created transforms.py
Created dataset.py
Created model.py
Created losses.py
Created metrics.py
Created train.py
Created test.py
All files have been generated successfully!
Make sure your dataset folder is uploaded to Colab.
Then, create a new cell and run: !python train.py


In [2]:
import os
import shutil
import glob
from tqdm import tqdm

# 1. Setup target directories
base_dir = '/content/dataset'
splits = ['train', 'val', 'test']

print("Creating unified dataset structure...")
for split in splits:
    os.makedirs(os.path.join(base_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(base_dir, split, 'masks'), exist_ok=True)

# ==========================================
# 2. Process archive (2)
# ==========================================
archive_path = "/content/drive/MyDrive/archive (2)"
print("\n--- Processing archive (2) ---")
if os.path.exists(archive_path):
    for split in splits:
        img_dir = os.path.join(archive_path, split, 'images')
        mask_dir = os.path.join(archive_path, split, 'masks')

        if not os.path.exists(img_dir):
            continue

        # Get all images (handling jpg, jpeg, png)
        images = []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            images.extend(glob.glob(os.path.join(img_dir, ext)))

        for img_path in tqdm(images, desc=f"archive(2) - {split}"):
            base_name = os.path.basename(img_path)
            name_no_ext = os.path.splitext(base_name)[0]

            # Find the corresponding mask (it might be .png or .jpg in the source)
            mask_path = None
            for ext in ('.png', '.jpg', '.jpeg'):
                temp_mask_path = os.path.join(mask_dir, name_no_ext + ext)
                if os.path.exists(temp_mask_path):
                    mask_path = temp_mask_path
                    break

            if mask_path:
                # Save both with the EXACT same extension (e.g. .jpg) to satisfy dataset.py
                dest_img = os.path.join(base_dir, split, 'images', base_name)
                dest_mask = os.path.join(base_dir, split, 'masks', base_name)

                shutil.copy(img_path, dest_img)
                shutil.copy(mask_path, dest_mask)
else:
    print("Could not find 'archive (2)' in Drive. Check the spelling.")

# ==========================================
# 3. Process CRACK500
# ==========================================
crack500_path = "/content/drive/MyDrive/CRACK500"
mapping = {'traincrop': 'train', 'valcrop': 'val', 'testcrop': 'test'}

print("\n--- Processing CRACK500 ---")
if os.path.exists(crack500_path):
    for c500_folder, split in mapping.items():
        src_dir = os.path.join(crack500_path, c500_folder)
        if not os.path.exists(src_dir):
            continue

        image_files = glob.glob(os.path.join(src_dir, '*.jpg'))

        for img_path in tqdm(image_files, desc=f"CRACK500 - {c500_folder}"):
            base_name = os.path.basename(img_path)
            name_no_ext = os.path.splitext(base_name)[0]

            # In CRACK500, masks are .png with the exact same name
            mask_path = os.path.join(src_dir, name_no_ext + '.png')

            if os.path.exists(mask_path):
                # Prefix to avoid naming collisions with archive(2)
                new_name = f"c500_{base_name}"

                dest_img = os.path.join(base_dir, split, 'images', new_name)
                dest_mask = os.path.join(base_dir, split, 'masks', new_name)

                shutil.copy(img_path, dest_img)
                shutil.copy(mask_path, dest_mask)
else:
    print("Could not find 'CRACK500' in Drive. Check the spelling.")

print("\n=======================================================")
print("Dataset merging completely finished! ")
print("Run the next cell with: !python train.py")

Creating unified dataset structure...

--- Processing archive (2) ---


archive(2) - test: 100%|██████████| 1119/1119 [07:43<00:00,  2.41it/s]



--- Processing CRACK500 ---


CRACK500 - traincrop: 0it [00:00, ?it/s]
CRACK500 - valcrop: 0it [00:00, ?it/s]
CRACK500 - testcrop: 0it [00:00, ?it/s]


Dataset merging completely finished! 
Run the next cell with: !python train.py


In [ ]:
!python train.py

Using device: cuda
config.json: 100% 156/156 [00:00<00:00, 725kB/s]

model.safetensors: downloading bytes:  92% 80.6M/87.3M [00:01<00:00, 77.0MB/s, 7.38MB/s  ]
model.safetensors: downloading bytes: 100% 80.6M/80.6M [00:01<00:00, 48.3MB/s, 7.61MB/s  ]
model.safetensors: reconstructing file: 100% 87.3M/87.3M [00:01<00:00, 52.3MB/s, 8.42MB/s  ]
Epoch 1/50 [Train]: 100% 474/474 [02:16<00:00,  3.48it/s, loss=0.62]
Epoch 1/50 [Val]: 100% 87/87 [00:05<00:00, 14.58it/s]

Epoch 1/50
Train Loss: 0.8384 | Validation Loss: 0.3809
IoU: 0.6808 | Dice: 0.8101 | Accuracy: 0.9684 | Precision: 0.7440

>>> Saved best_model.pth

Epoch 2/50 [Train]: 100% 474/474 [02:23<00:00,  3.29it/s, loss=0.385]
Epoch 2/50 [Val]: 100% 87/87 [00:06<00:00, 12.79it/s]

Epoch 2/50
Train Loss: 0.4929 | Validation Loss: 0.3315
IoU: 0.6846 | Dice: 0.8128 | Accuracy: 0.9695 | Precision: 0.7615

>>> Saved best_model.pth

Epoch 3/50 [Train]: 100% 474/474 [02:31<00:00,  3.14it/s, loss=0.349]
Epoch 3/50 [Val]: 100% 87/87 [00:06<00:

In [ ]:
!cp /content/last_model.pth "/content/drive/MyDrive/"
!cp /content/best_model.pth "/content/drive/MyDrive/"
print("Models safely saved to Google Drive!")

Models safely saved to Google Drive!


In [3]:
!cp "/content/drive/MyDrive/best_model.pth" /content/
!cp "/content/drive/MyDrive/last_model.pth" /content/
print("Models successfully loaded back into Colab!")

Models successfully loaded back into Colab!


In [4]:
!python test.py

Using device: cuda
config.json: 100% 156/156 [00:00<00:00, 642kB/s]

model.safetensors: downloading bytes:  82% 71.4M/87.3M [00:01<00:00, 81.5MB/s, 5.41MB/s  ]
model.safetensors: downloading bytes: 100% 80.6M/80.6M [00:01<00:00, 43.6MB/s, 7.52MB/s  ]
model.safetensors: reconstructing file: 100% 87.3M/87.3M [00:01<00:00, 47.2MB/s, 8.36MB/s  ]
Testing: 100% 280/280 [00:23<00:00, 12.00it/s]

--- Test Results ---
Test Loss: 0.3631
IoU:       0.6334
Dice:      0.7756
Accuracy:  0.9699
Precision: 0.8058
--------------------

Generating visualizations for 10 random samples...
Saved visualizations to test_visualizations.png


In [5]:
with open("resume_train.py", "w") as f:
    f.write('''import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import shutil

from dataset import CrackDataset
from transforms import get_train_transforms, get_val_test_transforms
from model import get_model
from losses import DiceBCELoss
from metrics import MetricTracker
from utils import set_seed, plot_training_curves

def train():
    # Hyperparameters
    DATASET_DIR = "dataset"
    IMAGE_SIZE = 512
    BATCH_SIZE = 4
    EPOCHS = 50
    START_EPOCH = 32  # Resuming from here
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 2
    PIN_MEMORY = True

    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Datasets and Dataloaders
    train_dataset = CrackDataset(DATASET_DIR, phase='train', transform=get_train_transforms(IMAGE_SIZE))
    val_dataset = CrackDataset(DATASET_DIR, phase='val', transform=get_val_test_transforms(IMAGE_SIZE))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # Model, Loss, Optimizer
    model = get_model().to(device)

    # Load previously saved weights
    if os.path.exists("last_model.pth"):
        print("Loading saved weights from last_model.pth...")
        model.load_state_dict(torch.load("last_model.pth", map_location=device))
    else:
        print("Error: last_model.pth not found in the current directory!")
        return

    criterion = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    metrics_tracker = MetricTracker()

    # Recovered history from Epochs 1 to 31
    best_iou = 0.7175

    history_train_loss = [
        0.8384, 0.4929, 0.4415, 0.4196, 0.4111, 0.3953, 0.3899, 0.3830, 0.3874, 0.3737,
        0.3728, 0.3686, 0.3742, 0.3626, 0.3564, 0.3565, 0.3561, 0.3566, 0.3495, 0.3505,
        0.3472, 0.3407, 0.3370, 0.3409, 0.3338, 0.3317, 0.3306, 0.3299, 0.3279, 0.3263, 0.3252
    ]

    history_val_loss = [
        0.3809, 0.3315, 0.3171, 0.2936, 0.2981, 0.2922, 0.2802, 0.2922, 0.2918, 0.2849,
        0.2909, 0.2915, 0.2862, 0.3023, 0.2890, 0.2844, 0.2874, 0.2888, 0.2978, 0.2807,
        0.2862, 0.3031, 0.2814, 0.2900, 0.2855, 0.2873, 0.2886, 0.2812, 0.2946, 0.2837, 0.2764
    ]

    history_val_iou = [
        0.6808, 0.6846, 0.6917, 0.7087, 0.7028, 0.7085, 0.7165, 0.7050, 0.7054, 0.7134,
        0.7086, 0.7077, 0.7106, 0.6978, 0.7097, 0.7120, 0.7099, 0.7102, 0.7035, 0.7148,
        0.7090, 0.6971, 0.7144, 0.7080, 0.7100, 0.7091, 0.7091, 0.7157, 0.7043, 0.7140, 0.7175
    ]

    # Training Loop
    for epoch in range(START_EPOCH, EPOCHS + 1):
        # ----------------- TRAIN -----------------
        model.train()
        train_loss_epoch = 0.0

        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
        for images, masks in loop:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss_epoch += loss.item() * images.size(0)
            loop.set_postfix(loss=loss.item())

        train_loss_epoch /= len(train_loader.dataset)
        history_train_loss.append(train_loss_epoch)

        # ----------------- VALIDATION -----------------
        model.eval()
        val_loss_epoch = 0.0
        metrics_tracker.reset()

        with torch.no_grad():
            loop_val = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]")
            for images, masks in loop_val:
                images = images.to(device)
                masks = masks.to(device)

                outputs = model(images)
                loss = criterion(outputs, masks)

                val_loss_epoch += loss.item() * images.size(0)
                metrics_tracker.update(outputs, masks)

        val_loss_epoch /= len(val_loader.dataset)
        iou, dice, accuracy, precision = metrics_tracker.get_metrics()

        history_val_loss.append(val_loss_epoch)
        history_val_iou.append(iou)

        # Print metrics
        print(f"\\nEpoch {epoch}/{EPOCHS}")
        print(f"Train Loss: {train_loss_epoch:.4f} | Validation Loss: {val_loss_epoch:.4f}")
        print(f"IoU: {iou:.4f} | Dice: {dice:.4f} | Accuracy: {accuracy:.4f} | Precision: {precision:.4f}\\n")

        # Save Models locally and backup to Drive
        torch.save(model.state_dict(), "last_model.pth")
        shutil.copy("last_model.pth", "/content/drive/MyDrive/last_model.pth")

        if iou > best_iou:
            best_iou = iou
            torch.save(model.state_dict(), "best_model.pth")
            shutil.copy("best_model.pth", "/content/drive/MyDrive/best_model.pth")
            print(">>> Saved best_model.pth (Local and Drive)\\n")

    plot_training_curves(history_train_loss, history_val_loss, history_val_iou, "full_training_curves.png")
    print("Training complete. Curves saved as full_training_curves.png.")

if __name__ == "__main__":
    train()
''')
print("Created resume_train.py with your full Epoch 1-31 history intact!")

Created resume_train.py with your full Epoch 1-31 history intact!


In [7]:
!python resume_train.py

Using device: cuda
Loading saved weights from last_model.pth...
Epoch 32/50 [Train]: 100% 474/474 [02:18<00:00,  3.42it/s, loss=0.259]
Epoch 32/50 [Val]: 100% 87/87 [00:06<00:00, 13.91it/s]

Epoch 32/50
Train Loss: 0.2854 | Validation Loss: 0.2875
IoU: 0.7126 | Dice: 0.8322 | Accuracy: 0.9738 | Precision: 0.8102

Epoch 33/50 [Train]: 100% 474/474 [02:24<00:00,  3.29it/s, loss=0.305]
Epoch 33/50 [Val]: 100% 87/87 [00:06<00:00, 12.94it/s]

Epoch 33/50
Train Loss: 0.2796 | Validation Loss: 0.2951
IoU: 0.7065 | Dice: 0.8280 | Accuracy: 0.9735 | Precision: 0.8163

Epoch 34/50 [Train]: 100% 474/474 [02:24<00:00,  3.29it/s, loss=0.276]
Epoch 34/50 [Val]: 100% 87/87 [00:06<00:00, 13.33it/s]

Epoch 34/50
Train Loss: 0.2807 | Validation Loss: 0.2913
IoU: 0.7090 | Dice: 0.8297 | Accuracy: 0.9734 | Precision: 0.8076

Epoch 35/50 [Train]: 100% 474/474 [02:24<00:00,  3.29it/s, loss=0.276]
Epoch 35/50 [Val]: 100% 87/87 [00:06<00:00, 14.00it/s]

Epoch 35/50
Train Loss: 0.2819 | Validation Loss: 0.2945

In [8]:
!cp /content/full_training_curves.png "/content/drive/MyDrive/"
print("Training graph safely copied to Google Drive!")

Training graph safely copied to Google Drive!


In [9]:
!cp /content/best_model.pth "/content/drive/MyDrive/"
!cp /content/last_model.pth "/content/drive/MyDrive/"
!cp /content/full_training_curves.png "/content/drive/MyDrive/"
!cp /content/test_visualizations.png "/content/drive/MyDrive/"

print("All models and graphs successfully saved to Google Drive!")

All models and graphs successfully saved to Google Drive!


In [ ]:
!pip install -q streamlit
!npm install localtunnel

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import cv2
import numpy as np
from PIL import Image
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ════════════════════════════════════════════════════════════════════════════
#  Page Config & Custom CSS
# ════════════════════════════════════════════════════════════════════════════
st.set_page_config(page_title="AI Crack Detection Analyzer", layout="wide")

st.markdown("""
<style>
.stat-box   { border:1px solid #333; border-radius:10px; padding:16px; background:#111; }
.stat-label { font-size:11px; text-transform:uppercase; letter-spacing:.06em; color:#888; }
.stat-value { font-size:26px; font-weight:600; margin-top:4px; }
.accent     { border-color:#4f8ef7; background:#0d1f40; }
.accent .stat-value { color:#4f8ef7; }
</style>
""", unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════════════════════
#  Load PyTorch Model
# ════════════════════════════════════════════════════════════════════════════
@st.cache_resource
def load_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights=None,
        classes=1,
        activation=None
    )
    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    model.to(device)
    model.eval()
    return model, device

model, device = load_model()

transform = A.Compose([
    A.Resize(height=512, width=512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# ════════════════════════════════════════════════════════════════════════════
#  Prediction & NOISE REDUCTION Logic
# ════════════════════════════════════════════════════════════════════════════
def predict_crack(img_bgr, threshold, min_area, opening_k, dilate_it):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 1. Prepare & Predict
    augmented = transform(image=img_rgb)
    input_tensor = augmented['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        prob_mask = torch.sigmoid(output).squeeze().cpu().numpy()

    # 2. Resize to display size
    prob_mask_resized = cv2.resize(prob_mask, (800, 600), interpolation=cv2.INTER_LINEAR)
    img_display = cv2.resize(img_rgb, (800, 600))

    # 3. Apply Threshold
    binary_mask = (prob_mask_resized > threshold).astype(np.uint8)

    # 4. STRICT NOISE REDUCTION: Morphological Opening
    # This specifically kills tiny isolated dots BEFORE we do anything else
    if opening_k > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (opening_k, opening_k))
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)

    # 5. Dilation (reconnects cracks that might have small gaps)
    if dilate_it > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        binary_mask = cv2.dilate(binary_mask, kernel, iterations=dilate_it)

    # 6. STRICT NOISE REDUCTION: Area Filtering
    # Completely wipes out any blobs smaller than our threshold
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    clean_mask = np.zeros_like(binary_mask)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            clean_mask[labels == i] = 1

    # Calculate stats
    if clean_mask.sum() > 0:
        confidence = prob_mask_resized[clean_mask == 1].mean()
    else:
        confidence = 0.99

    coverage = clean_mask.sum() / (800 * 600)
    has_crack = coverage > 0.0005

    # Generate Clean Overlay
    overlay = img_display.copy().astype(np.float32)
    overlay[clean_mask == 1] = overlay[clean_mask == 1] * 0.3 + np.array([40, 220, 120]) * 0.7
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    return img_display, clean_mask, overlay, has_crack, coverage, confidence

# ════════════════════════════════════════════════════════════════════════════
#  UI Layout
# ════════════════════════════════════════════════════════════════════════════
st.markdown("## AI Crack Detection Analyzer (Noise-Free)")
st.caption(
    "Powered by your custom-trained PyTorch DeepLabV3+ Model! "
    "Advanced noise-reduction active."
)

with st.sidebar:
    st.markdown("### AI & Noise Parameters")

    st.markdown("#### 1. AI Confidence")
    threshold = st.slider("Detection Threshold", 0.1, 0.9, 0.6, step=0.05,
                          help="0.6 forces the AI to be more strict, reducing false positive pebbles.")

    st.markdown("#### 2. Strict Noise Removal")
    min_area = st.slider("Minimum Blob Area (px)", 0, 2000, 400, step=50,
                         help="CRITICAL: Set high (~400) to delete all small dots and pebbles.")

    opening_k = st.slider("Morphological Opening (Despeckle)", 0, 7, 3, step=2,
                          help="Mathematically deletes tiny noise before processing.")

    st.markdown("#### 3. Crack Enhancement")
    dilate_it = st.slider("Dilate mask (thickness)", 0, 5, 2,
                          help="Makes the final continuous crack lines thicker.")

uploaded = st.file_uploader(
    "Upload images", type=["jpg", "jpeg", "png"],
    accept_multiple_files=True, label_visibility="collapsed",
)

if uploaded:
    results = []
    prog = st.progress(0, text="AI is Analyzing…")
    for i, f in enumerate(uploaded):
        pil_img = Image.open(f).convert("RGB")
        img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

        img_disp, mask, overlay, has_c, cov, conf = predict_crack(
            img_bgr, threshold, min_area, opening_k, dilate_it
        )

        results.append({
            "name": f.name, "img": img_disp, "mask": mask,
            "overlay": overlay, "has_crack": has_c,
            "coverage": cov, "confidence": conf
        })
        prog.progress((i + 1) / len(uploaded), text=f"Analyzed {i+1}/{len(uploaded)}")
    prog.empty()

    done = len(results)
    cracks = sum(1 for r in results if r["has_crack"])
    avg_conf = sum(r["confidence"] for r in results) / done if done > 0 else 0

    c1, c2, c3, c4 = st.columns(4)
    for col, label, value, acc in [
        (c1, "Images", str(done), False),
        (c2, "Analyzed", f"{done}/{done}", False),
        (c3, "Cracks detected", str(cracks), False),
        (c4, "Avg. AI Confidence", f"{avg_conf*100:.1f}%", True),
    ]:
        with col:
            cls = "stat-box accent" if acc else "stat-box"
            st.markdown(f'<div class="{cls}"><div class="stat-label">{label}</div>'
                        f'<div class="stat-value">{value}</div></div>', unsafe_allow_html=True)
    st.markdown("---")

    for idx, res in enumerate(results):
        st.markdown(f"#### {idx+1}. {res['name']}  ·  AI Confidence {res['confidence']*100:.1f}%")

        col_u, col_d = st.columns(2)
        with col_u:
            st.markdown("**Original Image**")
            st.image(res["img"], use_container_width=True)

        with col_d:
            tag = f"Crack detected · {res['coverage']*100:.2f}% area" if res["has_crack"] else "No crack detected"
            st.markdown(f"**Clean AI Prediction** — {tag}")
            st.image(res["overlay"], caption="Green AI Overlay", use_container_width=True)

        st.image(res["mask"] * 255, caption="Noise-Free Binary Mask", use_container_width=True)
        st.markdown("---")
else:
    st.markdown(
        '<div style="border:2px dashed #333;border-radius:12px;padding:48px;text-align:center;color:#666">'
        'Drag & drop images here, or click <b>Browse files</b> above.<br>'
        '<small>Upload an image to see the new noise-free results!</small>'
        '</div>', unsafe_allow_html=True
    )

In [ ]:
import urllib
print("Password/Enpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501